# Create Schemas and Tables 

## Overview
This notebook orchestrates the creation of all database schemas and tables for the business data platform.

**Execution Order**: Foundational domains first (customer, product) → Business processes (sales, finance) → Operations (inventory, supplychain)

**Prerequisites**: 
- Fabric lakehouse properly configured
- PySpark session active
- Schema notebooks available in `../schema/` directory

## Prepare Clean Environment

**⚠️ Warning**: Uncomment the lines below only if you want to completely reset all schemas and data.

- `truncate_all_tables`: Removes all data but keeps table structures
- `delete_all_schemas`: Drops all schemas and tables entirely

**Recommendation**: Run selectively for development/testing environments only.

In [ ]:
# %run truncate_all_tables
# %run delete_all_schemas

### Create schema and tables for customer domain

In [ ]:
%run model_customer

### Create schema and tables for product domain

In [ ]:
%run model_product

### Create schema and tables for sales domain 

In [ ]:
%run model_sales

### Create schema and tables for finance domain

In [ ]:
%run model_finance

In [ ]:
# Validation checkpoint - verify core schemas exist
try:
    databases = [db.name for db in spark.sql("SHOW DATABASES").collect()]
    core_schemas = ["customer", "product", "sales", "finance"]
    
    print("🔍 Schema Creation Validation:")
    for schema in core_schemas:
        if schema in databases:
            table_count = spark.sql(f"SHOW TABLES IN {schema}").count()
            print(f"   ✅ {schema}: {table_count} tables created")
        else:
            print(f"   ❌ {schema}: Schema not found!")
            
except Exception as e:
    print(f"❌ Validation error: {e}")
    print("   Please check that previous schema creations completed successfully")

### Create schema and tables for inventory domain

In [ ]:
%run model_inventory

### Create schema and tables for supplychain domain

In [ ]:
%run model_supplychain

## Final Validation & Summary

In [ ]:
# Complete schema and table validation
print("🎯 Complete Schema Creation Summary")
print("=" * 50)

try:
    databases = [db.name for db in spark.sql("SHOW DATABASES").collect()]
    all_schemas = ["customer", "product", "sales", "finance", "inventory", "supplychain"]
    total_tables = 0
    
    for schema in all_schemas:
        if schema in databases:
            tables_result = spark.sql(f"SHOW TABLES IN {schema}")
            table_count = tables_result.count()
            total_tables += table_count
            
            print(f"\n📊 {schema.upper()} SCHEMA:")
            print(f"   Status: ✅ Created")
            print(f"   Tables: {table_count}")
            
            # Show table names for verification
            if table_count > 0:
                table_names = [row.tableName for row in tables_result.collect()]
                for table in table_names:
                    print(f"     • {table}")
        else:
            print(f"\n📊 {schema.upper()} SCHEMA:")
            print(f"   Status: ❌ Missing")
    
    print(f"\n🎉 SUMMARY:")
    print(f"   Schemas created: {len([s for s in all_schemas if s in databases])}/{len(all_schemas)}")
    print(f"   Total tables: {total_tables}")
    print(f"   Platform ready for data loading!")
    
except Exception as e:
    print(f"❌ Validation failed: {e}")
    print("Please review the execution log for errors in schema creation.")

## Configuration Management

**New Feature**: Warehouse Configuration
- Warehouse locations are now configurable via `warehouses.json` in the input directory
- Replaces hardcoded warehouse locations with dynamic configuration
- Supports warehouse priorities, display names, and delivery location mapping
- Follows the same pattern as `suppliers.json` for consistency

**Configuration Files Required**:
- `suppliers.json` - Supplier master data and relationships
- `warehouses.json` - Warehouse locations, capacities, and display names